**Data Modeling Phase (Power BI)**

Implementation Plan

Goal: build a correct, stable Power BI data model that is ready for DAX and dashboards.

**Step 1 — Load Data into Power BI**

What to do:
1) Open Power BI Desktop

2) Load:
- analysis_ready_transactions.csv
- analysis_ready_customer_summary.csv

3) Use Import mode

4) Disable:
- Auto date/time (Options → Data Load)

Why:
- Prevent hidden date tables
- Full control over time logic

Result:
- Two raw tables loaded exactly as produced in Phase 2

**Step 2 — Create FactSales**

What to do:
1) Rename cleaned_transactions → FactSales

2) In Power Query:
- Keep only fields defined for FactSales
- Do not create calculated columns

3) Confirm data types:
- order_date → Date
- Customer ID → Whole number (nullable)
- line_revenue, order_revenue → Decimal

Do NOT:
- Aggregate anything
- Remove negative values
- Add business logic

Result:
- One transactional fact at invoice line grain

**Step 3 — Create FactCustomerSummary**

What to do:
1) Rename customer_summary → FactCustomerSummary

2) Keep table as-is

3) Validate:
- customerid matches FactSales type
- Date fields are Date type

Do NOT:
- Join it to FactSales
- Create time logic

Result:
- Snapshot fact at customer grain

**Step 4 — Create DimDate (Calendar Table)**

What to do:

1) Create a new table (DAX or Power Query)

2) Generate dates:
- From min(FactSales[order_date])
- To max(FactSales[order_date])

3) Add columns:

- year
- month
- year_month
- quarter
- week
- day_name
- is_weekend

Mark as:
- Date table

Result:
- Single authoritative time dimension

**Step 5 — Create DimCustomer**

What to do:

1) Duplicate FactCustomerSummary

2) Keep only:
- customerid
- first_purchase_date
- last_purchase_date
- customer_lifetime_days

3) Remove duplicates

4) Rename → DimCustomer

Why:
- Clean star schema
- Shared customer filtering

Result:
- One customer dimension

**Step 6 — Create DimProduct**

What to do:

1) Reference FactSales

2) Keep:
- StockCode
- Description

3) Remove duplicates

4) Rename → DimProduct

Result:
- One product dimension

**Step 7 — Create DimCountry**

What to do:

1) Reference FactSales

2) Keep:
- Country

3) Remove duplicates

4) Rename → DimCountry

**Step 8 — Define Relationships**

What to do (Model view):

1) Create relationships:
- DimDate[date] → FactSales[order_date]
- DimCustomer[Customer ID] → FactSales[Customer ID]
- DimCustomer[Customer ID] → FactCustomerSummary[Customer ID]
- DimProduct[StockCode] → FactSales[StockCode]
- DimCountry[Country] → FactSales[Country]

2) For all relationships:
- Cardinality: One-to-many
- Cross-filter: Single direction
- Active

Do NOT:
- Enable bi-directional filtering
- Create many-to-many joins

**Step 9 — Hide Technical Columns**

What to do:

1) Hide from report view:
- Surrogate or helper fields
- Repeated order-level fields (order_revenue, items, lines)
- Raw timestamps (InvoiceDate)
- Raw time breakdown fields (year, month, week in FactSales)

Why:
- Force correct slicing paths
- Prevent analyst mistakes later

**Step 10 — Model Validation (Mandatory)**

Before moving on:

Check:
- Slicing by date does not inflate customer metrics
- Summing line_revenue works as expected
- Returns reduce revenue correctly
- FactCustomerSummary metrics do not multiply
- No ambiguous relationships
- If something fails → fix model, not DAX.

**Data Modeling Phase 4 Output**

At the end you must have:
- FactSales
- FactCustomerSummary
- DimDate
- DimCustomer
- DimProduct
- DimCountry
- Clean star schema
- No visuals
- No DAX KPIs

**DAX Measures & KPI Definitions**

Goal

Create a canonical, reusable DAX measure layer that supports all business questions already answered in SQL and enables clean Power BI dashboards.

**Measure Design Principles (Lock First)**

Before writing any DAX, we lock a strict set of principles.
These rules prevent double counting, unstable KPIs, and “working but wrong” dashboards.

**Measures Only (No Calculated Columns)**

Rule:
- All business logic lives in measures
- No calculated columns for KPIs

Why:
- Measures respect filter context
- Columns do not
- Columns inflate data and break time analysis

**Measures Belong to Fact Tables**

Rule:
- Measures are created on:
    - FactSales
    - FactCustomerSummary

Never on dimensions

Why:
- Facts own numeric truth
- Dimensions only filter

Result:
- Clean semantic layer
- Predictable evaluation context

**One Business Concept = One Measure**

Rule:
- One measure answers one question
- No “do-everything” measures

Bad:
- RevenueAdjustedWithReturnsAndTax

Good:
- Gross Revenue
- Return Revenue
- Net Revenue

Why:
- Easier debugging
- Reusable logic
- Clear KPI definitions

**Additivity Must Be Explicit**

Rule:
- Measures are assumed additive by default
- Non-additive behavior must be intentional and documented

Examples:

Additive:
- Total Revenue
- Orders Count

Semi-additive:
- AOV
- Return Rate

Why:
- Prevents incorrect totals
- Forces awareness of aggregation behavior

**Never Aggregate Order-Level Fields Directly**

Rule:

Never SUM:
- order_revenue
- items
- lines

Why:
- These values repeat per invoice line
- Direct aggregation causes overcounting

Correct pattern:
- Aggregate by order_id first
- Then compute measures
- This rule is non-negotiable.

**Dimensions Must Be Filter-Only**

Rule:

Dimensions:
- filter facts
- never compute logic

Do NOT:
- Write measures on dimensions
- Use dimension columns for arithmetic

Why:
- Prevents circular dependencies
- Preserves star schema integrity

**Time Intelligence Uses DimDate Only**

Rule:
- All time-based logic must use DimDate
- Never use date fields from fact tables

Why:
- Single source of time truth
- Correct MoM / YoY behavior
- Avoids hidden date tables

**Measures Must Be Context-Safe**

Rule:
Measures must behave correctly when:
- sliced by date
- sliced by product
- sliced by customer
- combined slicers are applied

If a measure breaks:
- Fix the measure
- Do not patch with visuals

**Naming Convention (Mandatory)**

Rule:
- Measure names reflect business meaning
- No table prefixes

Examples:
- Total Revenue
- Net Revenue
- Orders Count
- Return Rate

Why:
- Business users read measures
- Tables are implementation details


**Core Revenue Measures (Foundation)**

This section defines the base revenue layer.

All measures are created on FactSales.

**Total Revenue**

Business meaning
Net monetary result of all sales including returns.

This must be the single source of truth for revenue.

DAX

Total Revenue :=
SUM ( FactSales[line_revenue] )

**Gross Revenue**

Business meaning
Revenue from positive sales only, ignoring returns.

Used to understand demand before returns.

DAX

Gross Revenue :=
CALCULATE (
    SUM ( FactSales[line_revenue] ),
    FactSales[is_positive_sale] = TRUE ()
)

**Return Revenue**

Business meaning
Revenue lost due to returns.

This measure remains negative by design.

DAX

Return Revenue :=
CALCULATE (
    SUM ( FactSales[line_revenue] ),
    FactSales[is_return_revenue] = TRUE ()
)

**Net Revenue (Explicit)**

Even though Total Revenue already represents net revenue,
we define this measure explicitly for clarity and documentation.

DAX

Net Revenue :=
[Gross Revenue] + [Return Revenue]

**What NOT to Do (Critical)**

Never create measures like:

SUM ( FactSales[order_revenue] )

or

SUM ( FactSales[Price] * FactSales[Quantity] )


Why:
- order_revenue is repeated per line
- Price × Quantity may break due to returns, corrections, or data fixes
- Business truth already exists in line_revenue

**Order & Basket Measures (Order-Safe DAX)**

Goal:
Create order-level and basket KPIs that are safe with line-grain data and cannot overcount.

All measures are created on FactSales.
No calculated columns.

**Orders Count**
Business meaning

Number of unique orders (invoices).

DAX

Orders Count :=
DISTINCTCOUNT ( FactSales[order_id] )

**Items Sold (Units)**
Business meaning

Total number of items sold, excluding returns.

DAX

Items Sold :=
CALCULATE (
    SUM ( FactSales[Quantity] ),
    FactSales[is_positive_sale] = TRUE ()
)

**Average Order Value (AOV)**

Business meaning

Average revenue per order.

DAX

Average Order Value :=
DIVIDE (
    [Net Revenue],
    [Orders Count]
)

**Average Items per Order**

Business meaning

Basket size in units.

DAX

Average Items per Order :=
DIVIDE (
    [Items Sold],
    [Orders Count]
)

**Orders with Returns (Count)**

Business meaning

Number of orders that contain at least one return.

DAX

Orders with Returns :=
CALCULATE (
    DISTINCTCOUNT ( FactSales[order_id] ),
    FactSales[is_return_revenue] = TRUE ()
)

**Return Order Rate**

Business meaning

Share of orders affected by returns.

DAX

Return Order Rate :=
DIVIDE (
    [Orders with Returns],
    [Orders Count]
)



**Customer KPIs (Snapshot-Safe DAX)**

Goal:
Create customer-level KPIs that:
- use FactCustomerSummary correctly
- are immune to date slicing (unless explicitly intended)
- do not multiply when combined with transactional data

All measures in this section are created on FactCustomerSummary.

**Customers Count**

Business meaning

Number of unique customers in the dataset.

DAX

Customers Count :=
COUNTROWS ( FactCustomerSummary )

**Total Customer Revenue**

Business meaning

Total revenue generated by all customers over the full period.

DAX

Total Customer Revenue :=
SUM ( FactCustomerSummary[total_revenue] )

**Average Customer Value (Lifetime)**

Business meaning

Average total revenue generated per customer.

DAX

Average Customer Value :=
DIVIDE (
    [Total Customer Revenue],
    [Customers Count]
)

**Average Customer Lifetime (Days)**

Business meaning

Average number of days between first and last purchase.

DAX

Average Customer Lifetime (Days) :=
AVERAGE ( FactCustomerSummary[customer_lifetime_days] )

**Active Customers**

Business meaning

Customers with more than one order (repeat customers).

DAX

Active Customers :=
CALCULATE (
    COUNTROWS ( FactCustomerSummary ),
    FactCustomerSummary[total_orders] > 1
)

**Average Orders per Customer**

Business meaning

How many orders an average customer places.

DAX

Average Orders per Customer :=
DIVIDE (
    SUM ( FactCustomerSummary[total_orders] ),
    [Customers Count]
)

**Repeat & Frequency Measures**

Goal:
Measure repeat behavior and purchase frequency in a way that:
- is mathematically correct
- matches SQL Phase logic
-does NOT inflate when slicing by date or product

We will use both facts, but each measure will respect grain boundaries.

**Repeat Customers Count**

Business meaning

Number of customers who placed more than one order.

DAX

Repeat Customers Count :=
CALCULATE (
    COUNTROWS ( FactCustomerSummary ),
    FactCustomerSummary[total_orders] > 1
)

**Repeat Customer Rate**

Business meaning

Share of customers who are repeat buyers.

DAX

Repeat Customer Rate :=
DIVIDE (
    [Repeat Customers Count],
    [Customers Count]
)

**Total Orders (Customer Perspective)**

This measure is needed for frequency calculations.

Create this on FactCustomerSummary:

DAX

Total Orders (Customers) :=
SUM ( FactCustomerSummary[total_orders] )

Average Purchase Frequency
Business meaning

**Average number of orders per customer.**

Business meaning

Average number of orders per customer.

DAX

Purchase Frequency :=
DIVIDE (
    [Total Orders (Customers)],
    [Customers Count]
)

**Repeat Customer Order Share (Advanced but Useful)**

Business meaning

How many orders are generated by repeat customers.

DAX

Orders from Repeat Customers :=
CALCULATE (
    SUM ( FactCustomerSummary[total_orders] ),
    FactCustomerSummary[total_orders] > 1
)

**5.6 Pareto (80/20) Measures**

Goal:
Identify revenue concentration and quantify how much revenue is generated by top customers.

All Pareto logic is built on FactCustomerSummary
because:
- customer grain is required
- anonymous customers must be excluded
- snapshot stability is critical

**Customer Revenue (Pareto Base)**

Even though revenue exists elsewhere, we define an explicit base measure for Pareto clarity.

Create this measure on FactCustomerSummary:

DAX

Customer Revenue :=
SUM ( FactCustomerSummary[total_revenue] )

**Customer Revenue Rank**

Business meaning

Rank customers from highest to lowest by total revenue.

DAX

Customer Revenue Rank :=
RANKX (
    ALL ( FactCustomerSummary[customerid] ),
    [Customer Revenue],
    ,
    DESC,
    DENSE
)

**Total Customer Revenue (Pareto Scope)**

Business meaning

We explicitly define total revenue for Pareto denominator.

DAX

Total Customer Revenue (Pareto) :=
CALCULATE (
    [Customer Revenue],
    ALL ( FactCustomerSummary )
)

**Customer Revenue Generated by 20% Top Spenders (Pareto Scope)**

Business meaning

We need this info, to check the Pareto's formula

DAX

Top 20% Customers Revenue :=
CALCULATE (
    [Customer Revenue],
    TOPN (
        ROUNDUP ( 0.2 * COUNTROWS ( FactCustomerSummary ), 0 ),
        ALL ( FactCustomerSummary ),
        [Customer Revenue],
        DESC
    )
)

**Top 20% Customer Spenders Revenue Share from Whole Revenue (Pareto Scope)**

DAX

Top 20% Revenue Share :=
DIVIDE (
    [Top 20% Customers Revenue],
    [Total Customer Revenue (Pareto)]
)

**Returns Impact Measures**

Goal:
Quantify how returns affect revenue and orders, in a way that is:
- correct with line-grain data
- stable under slicers
- easy to explain to business stakeholders

All measures are created on FactSales.

**Return Revenue (Already Created, Reuse)**

If you already created this reuse it.

DAX

Return Revenue :=
CALCULATE (
    SUM ( FactSales[line_revenue] ),
    FactSales[is_return_revenue] = TRUE ()
)


Meaning:
- Total revenue lost due to returns
- Value is negative
- No duplication. Do not recreate.

**Return Rate (Revenue-Based)**

Business meaning

What share of gross revenue is lost to returns.

This is the primary returns KPI.

DAX

Return Rate :=
DIVIDE (
    ABS ( [Return Revenue] ),
    [Gross Revenue]
)

**Orders with Returns (Reuse if Exists)**

If created reuse it:

DAX

Orders with Returns :=
CALCULATE (
    DISTINCTCOUNT ( FactSales[order_id] ),
    FactSales[is_return_revenue] = TRUE ()
)


Meaning:

Orders that contain at least one return line

**Return Order Rate**

Business meaning

What percentage of orders are affected by returns.

DAX

Return Order Rate :=
DIVIDE (
    [Orders with Returns],
    [Orders Count]
)

**Average Return Value per Return Order**

Business meaning

Average monetary impact per order that had returns.

DAX

Average Return Value per Order :=
DIVIDE (
    ABS ( [Return Revenue] ),
    [Orders with Returns]
)

**Net Revenue Loss from Returns (Explicit)**

Even though Net Revenue already reflects returns,
this measure isolates the loss explicitly.

DAX

Net Revenue Loss from Returns :=
ABS ( [Return Revenue] )

**Return Impact Index (Advanced but Simple)**
Business meaning

How severe returns are relative to order volume.

DAX

Return Impact Index :=
DIVIDE (
    [Net Revenue Loss from Returns],
    [Orders Count]
)

**Time Intelligence Measures**

Goal:
Add time-based comparisons that answer business questions like:

Are we growing month over month?

How does this month compare to last year?

Are orders following the same trend as revenue?

All measures are additive-safe, stable, and easy to validate.

**Revenue – Previous Month**

Business meaning

Revenue for the previous calendar month.

DAX

Revenue Previous Month :=
CALCULATE (
    [Net Revenue],
    DATEADD ( DimDate[Date], -1, MONTH )
)

**Month-over-Month Revenue Change (Absolute)**

Business meaning

How much revenue changed compared to the previous month.

DAX

Revenue MoM Change :=
[Net Revenue] - [Revenue Previous Month]

**Month-over-Month Revenue Growth %**

Business meaning

Relative growth compared to last month.

DAX

Revenue MoM Growth % :=
DIVIDE (
    [Revenue MoM Change],
    [Revenue Previous Month]
)

**Revenue – Same Month Last Year (YoY)**

Business meaning

Revenue in the same calendar month one year ago.

DAX

Revenue Same Month LY :=
CALCULATE (
    [Net Revenue],
    SAMEPERIODLASTYEAR ( DimDate[Date] )
)

**Year-over-Year Revenue Change**

Business meaning

Absolute revenue growth compared to the same month last year.

DAX

Revenue YoY Change :=
[Net Revenue] - [Revenue Same Month LY]

**Year-over-Year Revenue Growth %**

Business meaning

Relative YoY growth.

DAX

Revenue YoY Growth % :=
DIVIDE (
    [Revenue YoY Change],
    [Revenue Same Month LY]
)

Orders – Previous Month
Business meaning

**Orders count in the previous month.**

DAX

Orders Previous Month :=
CALCULATE (
    [Orders Count],
    DATEADD ( DimDate[Date], -1, MONTH )
)

**Orders MoM Growth %**

Business meaning

Change in order volume compared to last month.

DAX

Orders MoM Growth % :=
DIVIDE (
    [Orders Count] - [Orders Previous Month],
    [Orders Previous Month]
)
